## actualizar retiro telef 

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [2]:
query = f"""
	select NUMERO_DOCUMENTO as dni_cliente,color_final from DANTALION.dbo.Base_Maestra_Alfin_bk_Vigente
"""
df_maestra = pd.read_sql(query, engine_kishin)

df_maestra["dni_cliente"] = (
    df_maestra["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_maestra.columns.tolist())


['dni_cliente', 'color_final']


In [2]:
filename='fomato_agendas_alfin_credicash_2026.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_formato = pd.read_excel(filePath)
df_formato['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'
df_formato['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato['codigo_ejecutivo_id']='00000001'
df_formato['ejecutivo_target']='BOT'
df_formato['cdv_alfin_banco']='ROSA HONOR'

df_formato = df_formato.rename(columns={
    'monto': 'monto_solicitado'
})
df_formato["dni_cliente"] = (
    df_formato["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
# df_formato = df_formato.merge(
#     df_maestra,
#     on='dni_cliente',
#     how='left'
# )

df_formato['fecha_visita']='2026-07-27'

# Semilla opcional para reproducibilidad
# np.random.seed(123)

# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_formato))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato))

# Crear la columna
df_formato["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

df_formato['telefono_cliente']=df_formato['celular']
df_formato['dni_vendedor']=df_formato['ejecutivo_target']
df_formato['agencia_tienda']=df_formato['cod_agencia']
df_formato['operador']='TARGET'
df_formato['tipo_gestion']='Derivacion'

In [55]:
df_formato.drop_duplicates(subset=["dni_cliente"], inplace=True)



In [56]:
df_formato.shape

(1964, 19)

In [6]:
# df_formato = df_formato[
#     df_formato['agencia_atencion'].isin([
#         'SAN JUAN DE LURIG',
#         'ENMANCIPACION',
#         'PC HUANCAYO',
#         'TRUJ CENTRO',
#         'PC TACNA',
#         'PC HUARAZ',
#         'TRUJ AMERICA',
#         'AREQ CAYMA',
#         'AREQ PAMPILLA'
#     ])
# ]

#### Validar el nombre de la agencia

In [5]:
query = f"""
	select * from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)

set_correo = set(
    df_formato['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_correo']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

set()
set()


In [4]:
df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace("CAÃ‘ETE", "CAÑETE")
)

In [28]:
df_formato[df_formato['agencia_atencion']=='PC TACNA'].head()

,dni_cliente,nombre_cliente,celular,cod_agencia,agencia_atencion,fecha_visita,monto_solicitado,color_1,supervisor,canal_campo,codigo_ejecutivo_id,ejecutivo_target,cdv_alfin_banco,hora_visita,telefono_cliente,dni_vendedor,agencia_tienda,operador,tipo_gestion
57,00481993,COPARE HUANCA CECILIA DEL PILAR,931702300,738013 - PC TACNA,PC TACNA,2026-07-27,13100,VERDE OSCURO,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,16:15:00,931702300,BOT,738013 - PC TACNA,TARGET,Derivacion
68,45993184,Gustavo Mamani mamani,973366482,738013 - PC TACNA,PC TACNA,2026-07-27,16200,VERDE OSCURO,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,14:15:00,973366482,BOT,738013 - PC TACNA,TARGET,Derivacion
90,44842972,VIVIANA ELIZABETH PERALTA CHUCO DE PAUCAR,987562864,738013 - PC TACNA,PC TACNA,2026-07-27,22200,VERDE CLARO,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,16:15:00,987562864,BOT,738013 - PC TACNA,TARGET,Derivacion
363,04410070,VICTORIA HERMINIA TERESA SIFUENTES PIZA,955858440,738013 - PC TACNA,PC TACNA,2026-07-27,25000,VERDE CLARO,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,16:00:00,955858440,BOT,738013 - PC TACNA,TARGET,Derivacion
365,04417449,SAIRA COLANA JUAN CANCIO,952219125,738013 - PC TACNA,PC TACNA,2026-07-27,13200,VERDE CLARO,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,12:00:00,952219125,BOT,738013 - PC TACNA,TARGET,Derivacion


In [9]:
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'HUANCAYO', 'HUARAZ', 'TACNA'}
{'PC HUANCAYO', 'PC HUARAZ', 'PC TACNA'}


#### validar el codigo de agencia 

In [4]:
equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)

In [47]:
df_agencia[df_agencia['agencia_correo']=='TUMBES'].head()


,agencia_Formulario,agencia_base,agencia_base2,agencia_correo,correos
41,732000 - TUMBES,TUMBES,TUMBES,TUMBES,carlos.olivos@alfinbanco.pe


In [12]:
df_correo[df_correo['agencia_atencion']=='SAN JUAN DE MIRAFLORES'].head()


NameError: name 'df_correo' is not defined

In [12]:

set_correo = set(
    df_formato['agencia_tienda']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_Formulario']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'738381 - ENMANCIPACION', '739580 - ICA', '738360 - MOSHOQUEQUE', '737883 - COMAS', '738397 - AREQ CAYMA', '737896 - SAN JUAN DE LURIG', '739629 - ATE VITARTE', '739467 - HUACHO', '738224 - SAN JUAN DE MIRAFLORES', '738363 - CAJAMARCA', '735986 - JULIACA 2', '732243 - CAÑETE', '733825 - IQUITOS', '739483 - SAN MIGUEL', '738364 - TRUJ AMERICA', '734281 - CHICLAYO BALTA', '738371 - SAN MARTIN', '730109 - VENTANILLA', '737166 - MIRAFLORES', '739849 - LOS OLIVOS', '732000 - TUMBES', '734272 - CHIMBOTE', '734285 - PC HUARAZ', '738369 - PUENTE PIEDRA', '738382 - JESUS MARIA', '734280 - PC HUANCAYO', '734299 - CUSCO LA CULTURA', '739470 - HUARAL', '738252 - SANTA ANITA', '730879 - PAITA', '736568 - AREQ PAMPILLA', '738334 - PUCALLPA', '734264 - PISCO', '735996 - HUANUCO', '738391 - CHINCHA', '734265 - TRUJ CENTRO', '738013 - PC TACNA', '733824 - TARAPOTO', '734270 - SULLANA', '737490 - CASTILLA'}
set()


In [24]:
df_formato['agencia_tienda'] = (
    df_formato['agencia_atencion']
    .replace("CAÃ‘ETE", "CAÑETE")
)

In [ ]:
df_formato = df_formato[
    ~df_formato['agencia_tienda'].isin(['CASTILLA', 'AREQUIPA PAMPILLA', '734281 -  CHICLAYO BALTA', 'JESUS MARIA'])
].copy()

In [19]:
print(df_agencia["agencia_correo"].drop_duplicates().tolist())

['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']


In [ ]:
['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']MARIA 

In [50]:

equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)
# df_formato.drop_duplicates(subset=["dni"], inplace=True)

In [34]:
print(df_correo["agencia_atencion"].drop_duplicates().tolist())


['HUANUCO', 'HUANCAYO', 'SAN MIGUEL', 'CUSCO LA CULTURA', 'TACNA', 'TRUJILLO CENTRO', 'TARAPOTO', 'EMANCIPACION', 'CHIMBOTE', 'MIRAFLORES', 'TRUJILLO AMERICA', 'HUACHO', 'COMAS', 'CHICLAYO BALTA', 'SAN JUAN DE LURIGANCHO', 'CASTILLA', 'AREQUIPA PAMPILLA', 'SULLANA', 'VENTANILLA', 'MOSHOQUEQUE', 'JESUS MARIA', 'PUCALLPA', 'ATE VITARTE', 'LOS OLIVOS', 'VILLA MARIA 2', 'SANTA ANITA', 'CAJAMARCA', 'AREQUIPA CAYMA', 'PISCO', 'HUARAZ', 'SAN JUAN DE MIRAFLORES', 'CHINCHA', 'HUARAL', 'ICA', 'SAN MARTIN', 'JULIACA 2', 'VILLA EL SALVADOR 2', 'TUMBES', 'CAÑETE', 'PUENTE PIEDRA', nan, 'TE']


In [ ]:

df_correo[df_correo['dni_cliente']=='09704310'].head()

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
2190,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09704310,SALVADOR ALBERTO CHOQUE ALARCON,NaN,18000,930162239,MIRAFLORES,2026-07-15,13:30:00,MANUAL


In [6]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


# # Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()



# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())



C:\Users\DATA\AppData\Local\Temp\ipykernel_2292\3761581833.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [7]:
df_formato['retiro'] = (
    df_formato['dni_cliente'].isin(dni_retiro) |
    df_formato['celular'].isin(cel_retiro)
).astype(int)

In [8]:
df_formato=df_formato[df_formato['retiro']==0]

In [63]:
df_formato.count()

dni_cliente            1964
nombre_cliente         1964
celular                1964
cod_agencia            1964
agencia_atencion       1964
fecha_visita           1964
monto_solicitado       1964
color_1                1964
supervisor             1964
canal_campo            1964
codigo_ejecutivo_id    1964
ejecutivo_target       1964
cdv_alfin_banco        1964
hora_visita            1964
telefono_cliente       1964
dni_vendedor           1964
agencia_tienda         1964
operador               1964
tipo_gestion           1964
retiro                 1964
dtype: int64

In [47]:
df_formato.columns.tolist()

['dni_cliente',
 'nombre_cliente',
 'celular',
 'cod_agencia',
 'agencia_atencion',
 'fecha_visita',
 'monto_solicitado',
 'color',
 'supervisor',
 'canal_campo',
 'codigo_ejecutivo_id',
 'ejecutivo_target',
 'cdv_alfin_banco',
 'hora_visita',
 'telefono_cliente',
 'dni_vendedor',
 'agencia_tienda',
 'operador',
 'tipo_gestion',
 'retiro']

In [2]:
query = f"""
	SELECT dni_cliente FROM Alice.prospectos_envio_alfin 
    where estado='procesado'
    and fecha_envio<'2026-08-10'
"""
df_prospectos_envio_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where estado='ENVIADO'
	and DATE(fecha_envio) in('2026-08-04','2026-08-08')
    and fecha_envio<'2026-08-10'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)
df_seg=df_prospectos_correos_alfin.merge(df_prospectos_envio_alfin[['dni_cliente']],on='dni_cliente',how='inner')
df_seg.shape

(7260, 21)

In [3]:
df_seg = df_seg.drop_duplicates(subset="dni_cliente")
df_seg.shape

(4435, 21)

In [4]:
query = f"""
	SELECT * FROM Alice.prospectos_envio_alfin 
    where fecha_envio>='2026-08-01'
"""
df_prospectos_envio = pd.read_sql(query, engine_mysql)

In [5]:
df_prospectos_envio=df_prospectos_envio.merge(df_seg[['dni_cliente']],on='dni_cliente',how='inner')
df_prospectos_envio = df_prospectos_envio.drop_duplicates(subset="dni_cliente")
df_prospectos_envio.shape


(4435, 16)

In [6]:
df_correo=df_seg[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita','color']] .copy()
df_correo['tipo_carga']='MANUAL'

df_formulario=df_prospectos_envio[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,color,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,16525138,CARMEN MEGO DE LOZANO,20000.0,948021393,MOSHOQUEQUE,2026-08-08,0 days 11:15:00,VERDE OSCURO,MANUAL
5,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,16670483,WILSON ALEX ARROYO,14800.0,974737887,MOSHOQUEQUE,2026-08-09,0 days 17:45:00,VERDE OSCURO,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,80644018,LUIS ALBERTO ALCANTARA CAPU+æAY\t,940755112,734281 - CHICLAYO BALTA,2026-08-05,10000,Derivacion
1,BOT,TARGET,80580577,ROSA ELVIRA LOPEZ CAMPOS,901737058,734281 - CHICLAYO BALTA,2026-08-05,7100,Derivacion


In [8]:
fechas = pd.date_range("2026-08-11", "2026-08-12")

df_correo["fecha_visita"] = np.random.choice(fechas, size=len(df_correo))

In [9]:
# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_correo))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_correo))

# Crear la columna
df_correo["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

In [10]:
df=df_correo[["fecha_visita",'dni_cliente']].copy()

In [11]:
df_formulario = df_formulario.drop(
    columns=["fecha_visita"]
)

In [12]:
df_formulario=df_formulario.merge(df,on='dni_cliente',how='left')

In [13]:
df_correo.shape

(4435, 14)

In [14]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

4435

In [22]:
df_correo_pendiente=df_correo.copy()
df_formulario_pendiente=df_formulario.copy()

In [44]:
df_formulario_pendiente[df_formulario_pendiente['dni_cliente']=='80248715'].head()

,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion


In [45]:
df_formulario[df_formulario['dni_cliente']=='80248715'].head()


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
15,BOT,TARGET,80248715,YOLANDA MARIA OLIVERO AGUILAR,991554388,737166 - MIRAFLORES,2026-08-02,4700,Derivacion


In [ ]:

ruta_archivo = os.path.join(ruta_alfin, 'VER_11.csv')
# df_formulario_pendiente.to_csv(ruta_archivo, sep=';')

In [46]:
df_formulario_actualizado = pd.read_csv(ruta_archivo,sep=';')

In [ ]:
df_formulario_actualizado['fecha_visita']=df_formulario_actualizado['fecha_visita']

In [51]:
df_formulario_actualizado["fecha_visita"] = pd.to_datetime(
    df_formulario_actualizado["fecha_visita"],
    errors="coerce"
)

In [ ]:
df_formulario_actualizado

In [52]:
df_formulario_actualizado.head()

,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,48124718,CHOQUE CURO NOEMY LUZMERY,937616610,737870 - VILLA MARIA 2,2026-06-08,20000,Derivacion
1,BOT,TARGET,41494656,GUTIERREZ HEREDIA CARLOS ALBERTO,994746090,737870 - VILLA MARIA 2,2026-03-08,19000,Derivacion
2,BOT,TARGET,41298318,ROXANA CELIA ESPINOZA MALLCO,970551073,737870 - VILLA MARIA 2,2026-05-08,4900,Derivacion
3,BOT,TARGET,40865470,PEREZ ALVA EDWUAR RICARDO,931126400,737870 - VILLA MARIA 2,2026-03-08,9000,Derivacion
4,BOT,TARGET,10098109,FLOR DE MARIA PAZ,913923143,737870 - VILLA MARIA 2,2026-03-08,20000,Derivacion


In [48]:
df_formulario_actualizado['dni_cliente'] = (
    df_formulario_actualizado['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)      # deja solo números
    .replace('', pd.NA)                      # vacío -> NA
    .mask(lambda s: s.str.len() > 8, pd.NA)  # >8 dígitos -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)